# Notebook 02 — Data Cleaning & ETL Pipeline

**Objective:** Run the production ETL pipeline that transforms the raw VC dataset into a clean, analysis-ready file. This notebook now calls the same functions as `scripts/etl_pipeline.py`, so the notebook and CLI outputs stay perfectly aligned.

In [ ]:
import pandas as pd

from pathlib import Path
import os
import warnings

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path(
    os.environ.get(
        'PROJECT_ROOT',
        Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd(),
    )
).resolve()

def first_existing(*candidates: Path) -> Path:
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[0]

RAW_DATA_PATH = first_existing(
    PROJECT_ROOT / 'data' / 'raw' / 'investments_VC.csv',
    PROJECT_ROOT / 'investments_VC.csv',
    Path('/content/investments_VC.csv'),
)

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import files as _colab_files
    IN_COLAB = True
except ImportError:
    _colab_files = None
    IN_COLAB = False

def maybe_download(path: Path) -> None:
    if IN_COLAB:
        _colab_files.download(str(path))

from scripts.etl_pipeline import build_clean_dataset, etl_log_to_dataframe, save_etl_log, save_processed

OUTPUT_CSV = OUTPUT_DIR / 'startups_cleaned.csv'
OUTPUT_LOG = OUTPUT_DIR / 'etl_run_log.csv'

print(f'Raw data path resolved to: {RAW_DATA_PATH}')
print(f'Clean data will be saved to: {OUTPUT_CSV}')

In [ ]:
df = build_clean_dataset(RAW_DATA_PATH)
print(f'\nClean dataset ready: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head(3)

In [ ]:
print('=' * 60)
print('FINAL DATASET QUALITY REPORT')
print('=' * 60)
print(f'\nShape:   {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Memory:  {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')
print('\n── Status distribution (target variable) ──')
status_dist = df['status'].value_counts()
for status, count in status_dist.items():
    pct = count / len(df) * 100
    print(f'  {status:12s}: {count:>6,}  ({pct:.1f}%)')

print('\n── Failure rate (closed) by funding tier ──')
print((df.groupby('funding_tier')['is_closed'].mean() * 100).round(2).sort_values(ascending=False).to_string())

print('\n── Remaining missing values in key columns ──')
key_cols = ['status', 'funding_total_usd', 'country_code', 'founded_year', 'funding_rounds', 'is_closed', 'primary_category']
print(df[key_cols].isnull().sum().to_string())

In [ ]:
etl_df = etl_log_to_dataframe()
print('=' * 80)
print('ETL PIPELINE AUDIT LOG')
print('=' * 80)
print(etl_df[['step', 'description', 'rows_before', 'rows_after', 'rows_dropped', 'detail']].to_string(index=False))
etl_df.head()

In [ ]:
outputs = save_processed(df, OUTPUT_CSV)
save_etl_log(OUTPUT_LOG)

print(f"✅ Cleaned dataset saved to: {outputs['csv']}")
print(f"✅ Missing report saved to: {outputs['missing_report']}")
print(f"✅ Metadata saved to: {outputs['metadata']}")
print(f"✅ ETL log saved to: {OUTPUT_LOG}")

maybe_download(outputs['csv'])
if not IN_COLAB:
    print('ℹ️ Not running on Colab — files saved locally.')

## Cleaning Takeaways

- The notebook and the standalone ETL script now share the same code path, which removes notebook-vs-script drift.
- All cleaned outputs are written to `data/processed/`, including `startups_cleaned.csv`, `missing_report.csv`, `etl_metadata.json`, and `etl_run_log.csv`.
- Colab downloads are optional and guarded, so the notebook runs safely in local Jupyter as well.